# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Emeka-techDev/ml-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

Selected Lane: Lane 2
Reason : it helps assist in decision making on what pages needs review first and provide a reasons why the pages should be reviewed.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision the work improves?**

This work assist in prioritizing pages that need review and saving large amount of time and limited resources that would go into impractical review of every page

**Who acts on it?**

Content writer and website reviewers

**Cost of wrong recommendation?**

time and resources are lost reviewing and updating the wrong site, site that actually need update and review might be ommitted, overall visible of the platform does not change


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [1]:
import pandas as pd

url = "https://huggingface.co/datasets/FlyRank/internship-starter/resolve/main/content_refresh_anonymized.csv"

df = pd.read_csv(url)

df.head()

print(f"Unique Number of pages: {df['content_id'].nunique():,}")

df.shape
print(
    f"Declining pages: {df['is_declining'].sum():,} "
    f"out of {len(df):,}"
)

# declining_rate = df["is_declining"].mean() * 100

# print(f"Declining pages: {declining_rate:.1f}%")
print(f"Total search volume: {df['search_volume'].sum():,.0f}")

Unique Number of pages: 30,000
Declining pages: 5,563 out of 30,000
Total search volume: 4,374,350


The starter file contains 30,000 pages, which makes reviewing each page individually impractical and creates a clear need for prioritization. Of those pages, 5,563 (18%) are declining, this suggest a meaningful subset of the content require review. The dataset contain a total search volume of 4,374,350 which indicates that the content represents a substantial amount of search demand, making the question which page should be reviewed first substantial valuable.
  

**BASELINE SETUP**



In [10]:
import numpy as np

# Signal 1: Recent performance decline

df["impression_change_pct"] = (
    (df["impressions_last_30d"] - df["impressions_prev_30d"])
    / df["impressions_prev_30d"].replace(0, np.nan)
) * 100

df["decline_signal"] = df["impression_change_pct"] < -20


# Signal 2: Declining + meaningful demand

df["has_demand"] = df["search_volume"].fillna(0) >= 20

df["declining_demand_score"] = (
    df["decline_signal"] &
    df["has_demand"]
).astype(int) * 3

# Signal 3: Stale + visible

df["stale_visible_score"] = (
    (df["days_since_last_update"] >= 91) &
    (df["impressions_90d"] > 0)
).astype(int) * 2


# Signal 4: Page-one decay risk

page_one = df["position_tier"].isin(
    ["top_3", "page_1", "striking"]
)

df["page_one_decay_score"] = (
    page_one &
    df["decline_signal"]
).astype(int) * 2


# Signal 5: CTR review context

ctr_threshold = df["ctr"].quantile(0.25)

df["ctr_review_score"] = (
    page_one &
    (df["ctr"] <= ctr_threshold)
).astype(int) * 1


# Final baseline score

df["baseline_score"] = (
    df["declining_demand_score"]
    + df["stale_visible_score"]
    + df["page_one_decay_score"]
    + df["ctr_review_score"]
)

baseline_queue = df.sort_values(
    "baseline_score",
    ascending=False
)

baseline_queue[
    [
        "content_id",
        "baseline_score",
        "search_volume",
        "impression_change_pct",
        "days_since_last_update",
        "position_tier",
        "ctr"
    ]
].head(20)

,content_id,baseline_score,search_volume,impression_change_pct,days_since_last_update,position_tier,ctr
21804,content_f31f9d523855,8,20.0,-100.000000,104,page_1,0.0
28238,content_0e2a59cb793e,8,260.0,-100.000000,104,page_1,0.0
6104,content_f1743a78cb16,8,20.0,-100.000000,92,top_3,0.0
1253,content_b47804a4832a,8,30.0,-58.196721,104,page_1,0.0
6839,content_e25886a96f6c,8,50.0,-100.000000,144,page_1,0.0
22556,content_eb299374f25e,8,40.0,-64.583333,104,striking,0.0
27699,content_6d077b964b59,8,70.0,-30.000000,104,striking,0.0
28269,content_8e329558bb9e,8,40.0,-65.748031,104,page_1,0.0
27092,content_96fedf694731,8,20.0,-21.827411,104,striking,0.0
27090,content_0f64f57f11ee,8,20.0,-75.555556,104,striking,0.0


In [2]:
#Signal 1 -- Declining
df["declining_score"] = df["is_declining"].astype(int) * 3

# Signal 2 -- Staleness
df["stale_score"] = (df["days_since_last_update"] > 180).astype(int) * 2
median_search_volume = df["search_volume"].median()

#Signal 3 -- Search Demand
df["demand_score"] = (df["search_volume"] > median_search_volume).astype(int) * 2

#Signal 4 -- Ranking Opportunity
position_points = {
    "page_1" : 2,
    "top_3": 2,
    "striking": 1,
    "page_3_5": 1,
    "deep": 0
}

df["position_score"] = (df["position_tier"].map(position_points).fillna(0))

df["baseline_score"] = (df["declining_score"] + df["stale_score"] + df["demand_score"] + df["position_score"])

baseline_queue = df.sort_values(by="baseline_score", ascending=False)

# baseline_queue.head(100)
baseline_queue[
    [
        "content_id",
        "baseline_score",
        "is_declining",
        "days_since_last_update",
        "search_volume",
        "position_tier"
    ]
].head(20)


,content_id,baseline_score,is_declining,days_since_last_update,search_volume,position_tier
6679,content_60e82604d07a,7,True,104,20.0,page_1
22244,content_38ca5024c34b,7,True,104,70.0,page_1
837,content_ebab0744ca5a,7,True,20,260.0,page_1
13286,content_5258796a68ea,7,True,104,210.0,page_1
3103,content_e32ec994d05a,7,True,14,50.0,page_1
27842,content_84e25eb7e6cd,7,True,22,210.0,page_1
27840,content_ced8288da781,7,True,20,30.0,page_1
9392,content_a05de90216c2,7,True,7,90.0,page_1
9814,content_a817704b744d,7,True,104,20.0,page_1
9815,content_b636045f0a0a,7,True,20,40.0,page_1


**Select Feature to use in training model**

In [3]:
print(pd.crosstab(
    df["trend_direction"],
    df["is_declining"],
    normalize="index"
))

is_declining        False     True 
trend_direction                    
down             0.657914  0.342086
flat             1.000000  0.000000
new              1.000000  0.000000
stable           1.000000  0.000000
up               1.000000  0.000000


The trend direction would be from the feature training data because the exploratory analysis showed that is is strongly aligned with the is_declining target and could leak information about the target into the model

In [4]:
print(
    df.groupby("position_tier")["is_declining"]
    .agg(["count", "mean"])
    .sort_values("mean", ascending=False)
)

               count      mean
position_tier                 
page_3_5        7242  0.224938
page_1         11814  0.196716
striking        7304  0.189348
top_3           2321  0.080569
deep            1319  0.030326


There is clearly an association between search position and declining status. however avg_position contains more information and we would use it instead of position_tier


In [6]:
print(
    df.groupby("freshness_tier")["is_declining"]
    .agg(["count", "mean"])
)

                count      mean
freshness_tier                 
0-30            20480  0.159619
181+              174  0.063218
31-90             175  0.102857
91-180           9171  0.246974


Freshness appear to contain useful information, but it's relationship with decline is not simply linear (this non-linearity would make the tree-based models outperform a simple linear model)

In [7]:
print(
    df.groupby("is_declining")[
        [
            "search_volume",
            "days_since_last_update",
            "ctr",
            "avg_position",
            "word_count",
            "engagement_rate"
        ]
    ].median()
)

              search_volume  days_since_last_update   ctr  avg_position  \
is_declining                                                              
False                  10.0                    20.0  0.00          10.7   
True                   10.0                    22.0  0.16          11.2   

              word_count  engagement_rate  
is_declining                               
False             2835.0              0.0  
True              3059.0              0.0  


### **Train model**

**STEP 1 : Create the feature**

In [8]:
# create features
features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

X = df[features].copy()
y = df["is_declining"].astype(int)

**STEP 2: split the data**


In [9]:
#Split the data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

**STEP 3: handle missing values (using a preprocessing pipeline)**


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scalar", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

**STEP 4: Train Model**

In [ ]:
model.fit(X_train, y_train)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scalar', StandardScaler()),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [ ]:
#Logisctic Regresion (lr) test score

lr_test_scores = model.predict_proba(X_test)[:, 1]

print(lr_test_scores)

[7.52793154e-01 5.78905859e-01 3.89265041e-01 ... 6.49393369e-06
 7.69069500e-01 3.47465449e-01]


In [ ]:
import numpy as np

results = X_test.copy()

results["actual_declining"] = y_test.values
results["logistic_regression_model_score"] = lr_test_scores

results = results.sort_values("logistic_regression_model_score", ascending=False)
top_20 = results.head(20)

precision_at_20 = top_20["actual_declining"].mean()
print(f"Precision@20: {precision_at_20:.3f}")

baseline_top_20 = baseline_queue.head(20)
baseline_precision_at_20 = baseline_top_20["is_declining"].mean()
print(f"Baseline Precision@20: {baseline_precision_at_20:.3f}")

Precision@20: 0.300
Baseline Precision@20: 1.000


6 of the top 20 pages selected by the model were actually labelled as declining

In [ ]:
from sklearn.metrics import average_precision_score

lr_ap = average_precision_score(
    y_test,
    lr_test_scores
)

print(f"Logistic regression Average Precision: {ap:.3f}")

#baseline average precision score
baseline_ap = average_precision_score(
    baseline_queue["is_declining"],
    baseline_queue["baseline_score"]
)

print(f"baseline Average Precision: {baseline_ap:.3f}")
#

Average Precision: 0.283
baseline Average Precision: 0.848


**Decision Tree model**

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('classifier', DecisionTreeClassifier(
        max_depth=5,
        random_state=42,
        class_weight="balanced"
    ))
])



In [ ]:
tree_model.fit(X_train, y_train)

tree_scores = tree_model.predict_proba(X_test)[:, 1]

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.